[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.0 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [20]:
# ✏️ YOUR IMPLEMENTATION HERE

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        # Initialize projections
        d_head = d_model // num_heads
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.d_model = d_model
        self.d_head = d_head
        self.scale = math.sqrt(d_head)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_kv_heads * d_head)
        self.W_v = nn.Linear(d_model, num_kv_heads * d_head)
        self.W_o = nn.Linear(d_model, d_model)

    def _split_head(self, x, num_heads, d_head):
        B, N, D = x.shape
        return x.view(B, N, num_heads, d_head).transpose(1, 2)

    def _merge_heads(self, x):
        B, num_heads, N, d_head = x.shape
        return x.transpose(1, 2).contiguous().view(B, N, num_heads * d_head)

    def forward(self, x):
        # Self-attention with grouped KV
        q = self._split_head(self.W_q(x), self.num_heads, self.d_head)
        k = self._split_head(self.W_k(x), self.num_kv_heads, self.d_head)
        v = self._split_head(self.W_v(x), self.num_kv_heads, self.d_head)
        n_repeats = self.num_heads // self.num_kv_heads
        k = k.repeat_interleave(n_repeats, dim=1)
        v = v.repeat_interleave(n_repeats, dim=1)
        score = torch.softmax(q @ k.transpose(-2, -1) / self.scale, dim=-1)
        weights_split = score @ v
        return self._merge_heads(weights_split)


In [21]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
Output shape: torch.Size([2, 6, 32])


In [22]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (3.7ms)
  ✅ [2/5] nn.Linear with correct shapes (1.5ms)
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (1.6ms)
  ✅ [4/5] KV heads are shared correctly (1.4ms)
  ✅ [5/5] Gradient flow (2.3ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (10.4ms total)
  Progress saved. Run status() to see your dashboard.

